# 04 — PEFT LoRA Fine-Tuning of CatVTON on Pakistani Stitched Kurtas
**Engineered for Free Colab Tesla T4 (16 GB VRAM):**
- CatVTON uses concatenation-only self-attention (no heavy ReferenceNet!).
- Attaching **PEFT LoRA (`r=16`)** onto UNet self-attention (`to_q`, `to_k`, `to_v`, `to_out.0`) reduces trainable weights to **~4.2M params** with `fp16` + gradient checkpointing + 8-bit AdamW.
- Trains on paired Pakistani kurta flat-lays + person photos with our 4-class kurta masks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q diffusers transformers accelerate peft bitsandbytes pillow tqdm

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb
from torchvision import transforms

device = 'cuda'
RESEARCH_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research')
LORA_OUT = RESEARCH_ROOT / '03_CATVTON_LORA_RUNS' / 'catvton_pk_lora_r16'
LORA_OUT.mkdir(parents=True, exist_ok=True)

# Base SD1.5-Inpainting backbone used by CatVTON
BASE_ID = 'booksforcharlie/stable-diffusion-inpainting'
vae = AutoencoderKL.from_pretrained(BASE_ID, subfolder='vae', torch_dtype=torch.float16).to(device).eval()
vae.requires_grad_(False)

unet = UNet2DConditionModel.from_pretrained(BASE_ID, subfolder='unet', torch_dtype=torch.float16).to(device)
unet.enable_gradient_checkpointing()

# Attach LoRA strictly to self-attention projections (CatVTON design principle)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
)
# Bypass torchao conflict in peft
import sys
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
for m in list(sys.modules.values()):
    if hasattr(m, 'is_torchao_available'):
        try:
            m.is_torchao_available = lambda: False
        except Exception:
            pass

unet = get_peft_model(unet, lora_cfg)
unet.print_trainable_parameters()

noise_scheduler = DDPMScheduler.from_pretrained(BASE_ID, subfolder='scheduler')
optimizer = bnb.optim.AdamW8bit(unet.parameters(), lr=1e-4, weight_decay=1e-2)
print('CatVTON LoRA ready inside 16GB T4 VRAM budget!')


In [ ]:
# Paired Dataset Loader expecting:
#   paired_train/person/*.jpg
#   paired_train/garment/*.jpg
#   paired_train/mask/*.png  (white = kurta region to inpaint, black = preserve face/bottom/dupatta)
TRAIN_DIR = RESEARCH_ROOT / '01_PK_VTON_DATASET' / 'paired_train'

img_tf = transforms.Compose([
    transforms.Resize((512, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
msk_tf = transforms.Compose([
    transforms.Resize((512, 384), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

class PairedPKVTONDataset(Dataset):
    def __init__(self, root: Path):
        self.person_files = sorted((root / 'person').glob('*.*')) if (root / 'person').exists() else []
        self.root = root
    def __len__(self):
        return len(self.person_files)
    def __getitem__(self, idx):
        p_path = self.person_files[idx]
        stem = p_path.stem
        g_path = next((self.root / 'garment').glob(f'{stem}.*'))
        m_path = next((self.root / 'mask').glob(f'{stem}.*'))
        return {
            'person': img_tf(Image.open(p_path).convert('RGB')),
            'garment': img_tf(Image.open(g_path).convert('RGB')),
            'mask': (msk_tf(Image.open(m_path).convert('L')) > 0.5).float(),
        }

ds = PairedPKVTONDataset(TRAIN_DIR)
print(f'Found {len(ds)} paired Pakistani training samples in {TRAIN_DIR}')

if len(ds) > 0:
    loader = DataLoader(ds, batch_size=1, shuffle=True)
    GRAD_ACCUM = 8
    MAX_STEPS = 1000
    step = 0
    unet.train()
    empty_cond = torch.zeros((1, 77, 768), device=device, dtype=torch.float16)

    while step < MAX_STEPS:
        for batch in loader:
            person = batch['person'].to(device, dtype=torch.float16)
            garment = batch['garment'].to(device, dtype=torch.float16)
            mask = batch['mask'].to(device, dtype=torch.float16)

            with torch.no_grad():
                person_lat = vae.encode(person).latent_dist.sample() * vae.config.scaling_factor
                masked_person = person * (1.0 - mask)
                masked_lat = vae.encode(masked_person).latent_dist.sample() * vae.config.scaling_factor
                garment_lat = vae.encode(garment).latent_dist.sample() * vae.config.scaling_factor
                mask_lat = F.interpolate(mask, size=person_lat.shape[-2:], mode='nearest')

            # CatVTON spatial concatenation along width: [person | garment]
            target_lat = torch.cat([person_lat, garment_lat], dim=-1)
            cond_lat = torch.cat([masked_lat, garment_lat], dim=-1)
            cond_mask = torch.cat([mask_lat, torch.zeros_like(mask_lat)], dim=-1)

            noise = torch.randn_like(target_lat)
            t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device=device).long()
            noisy_lat = noise_scheduler.add_noise(target_lat, noise, t)

            unet_in = torch.cat([noisy_lat, cond_mask, cond_lat], dim=1)
            pred_noise = unet(unet_in, t, encoder_hidden_states=empty_cond).sample

            # Compute loss strictly on the person half
            w_half = person_lat.shape[-1]
            loss = F.mse_loss(pred_noise[..., :w_half], noise[..., :w_half]) / GRAD_ACCUM
            loss.backward()

            if (step + 1) % GRAD_ACCUM == 0:
                optimizer.step()
                optimizer.zero_grad()

            step += 1
            if step % 100 == 0:
                print(f'Step {step}/{MAX_STEPS} | loss={loss.item()*GRAD_ACCUM:.4f}')
            if step % 250 == 0 or step >= MAX_STEPS:
                unet.save_pretrained(LORA_OUT / f'step_{step}')
            if step >= MAX_STEPS:
                break
    print('Saved final LoRA weights to:', LORA_OUT)
else:
    print('Populate paired_train/{person,garment,mask} from Notebook 01 & 02 to launch LoRA training!')